# Lab 11: Planner-Coder-Verifier Loop (DS-STAR Core)

**Navigation** : [Lab 10 <<](Lab10-File-Analyzer.ipynb) | [Index](../../README.md) | [>> Lab 12](Lab12-DS-Star-Workshop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter la boucle itérative Planner-Coder-Verifier de DS-STAR
2. Créer un système multi-agents pour l'analyse de données
3. Gérer les échecs et raffinements automatiques
4. Orchestrer plusieurs composants LLM en pipeline

### Prérequis
- Lab 10 (File Analyzer) complété
- Connaissance des patterns d'agents
- Configuration multi-provider active

### Durée estimée : 45-60 minutes

## 1. Architecture Planner-Coder-Verifier

> **Repères bibliographiques.** Cette boucle itérative Planner → Coder → Executor → Verifier est un cas particulier du paradigme **ReAct** (Reasoning + Acting) de S. Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, arXiv:2210.03629, ICLR 2023, où l'agent alterne raisonnement et actions observables. L'étape de *Verifier* qui renvoie vers le Coder en cas d'échec (raffinement automatique) suit le principe d'**auto-réflexion** formalisé par N. Shinn et al., *Reflexion: Language Agents with Verbal Reinforcement Learning*, arXiv:2303.11366, NeurIPS 2023. Le découpage en rôles spécialisés (Planner / Coder / Verifier) est l'architecture multi-agent déterministe retenue par DS-STAR (Guo et al., arXiv:2509.21825, 2025).
```
Question --> [PLANNER] --> Plan
                      |
                      v
                [CODER] --> Code
                      |
                      v
                [EXECUTOR] --> Result
                      |
                      v
                [VERIFIER] --> Success/Retry
```

### Schema : la boucle Planner-Coder-Verifier

Chaque etape transforme l'entree de la precedente ; le Verifier conclut sur un succes ou declenche un nouvel essai (retry).

```mermaid
flowchart TD
    Q["Question"] --> P["PLANNER (genere le Plan)"]
    P --> C["CODER (genere le Code)"]
    C --> E["EXECUTOR (produit le Result)"]
    E --> V["VERIFIER"]
    V --> SR["Success / Retry"]
```

## 2. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass
from enum import Enum

from config import get_settings
from utils import LLMClient

print("Imports OK : json, re, pandas, numpy, dataclasses, Enum, config, utils")

Imports OK : json, re, pandas, numpy, dataclasses, Enum, config, utils


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 3. Data Classes

In [1]:
@dataclass
class Plan:
    steps: List[str]
    reasoning: str

@dataclass
class ExecutionResult:
    success: bool
    output: str
    error: Optional[str] = None
    code: Optional[str] = None

class VerificationStatus(Enum):
    SUCCESS = 'success'
    NEEDS_REFINEMENT = 'needs_refinement'
    FAILED = 'failed'

print("Dataclasses definies : Plan, ExecutionResult, VerificationStatus (SUCCESS, NEEDS_REFINEMENT, FAILED)")

Dataclasses definies : Plan, ExecutionResult, VerificationStatus (SUCCESS, NEEDS_REFINEMENT, FAILED)


## 4. Planner Agent

In [1]:
class Planner:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def create_plan(self, question: str, context: str) -> Plan:
        prompt = f"""Tu es un planificateur d'analyse de donnees.

CONTEXTE DES FICHIERS:
{context}

QUESTION: {question}

Genere un plan d'analyse en 3-5 etapes. Pour chaque etape, decris:
1. L'objectif
2. La methode
3. Le resultat attendu

Format de sortie:
REASONING: [ton raisonnement]
STEPS:
1. [etape 1]
2. [etape 2]
...

Plan:"""

        response = self.llm.generate(prompt, temperature=0.3)

        # Parse response
        steps = []
        reasoning = ""
        lines = response.split('\n')
        in_steps = False

        for line in lines:
            if line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
            elif line.startswith('STEPS:'):
                in_steps = True
            elif in_steps and re.match(r'^\d+\.', line.strip()):
                steps.append(re.sub(r'^\d+\.\s*', '', line.strip()))

        return Plan(steps=steps, reasoning=reasoning)

print("Classe Planner definie : decomposition de questions en plans d'analyse (3-5 etapes)")

Classe Planner definie : decomposition de questions en plans d'analyse (3-5 etapes)


## 5. Coder Agent

In [1]:
class Coder:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_code(self, plan: Plan, context: str) -> str:
        steps_text = '\n'.join(f"{i+1}. {s}" for i, s in enumerate(plan.steps))

        prompt = f"""Tu es un programmeur Python expert. Genere du code pour executer ce plan.

CONTEXTE:
{context}

PLAN:
{steps_text}

Genere UNIQUEMENT du code Python executable entre balises ```python ... ```
Le DataFrame est disponible dans la variable 'df'.
Utilise print() pour afficher les resultats.

Code:"""

        response = self.llm.generate(prompt, temperature=0.2)

        # Extract code
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        if match:
            return match.group(1).strip()
        return response

print("Classe Coder definie : generation de code Python a partir d'un plan d'analyse")

Classe Coder definie : generation de code Python a partir d'un plan d'analyse


## 6. Executor

In [1]:
class Executor:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.namespace = {'df': df, 'pd': pd, 'np': np, 'print': print}

    def execute(self, code: str) -> ExecutionResult:
        import sys
        from io import StringIO

        old_stdout = sys.stdout
        sys.stdout = StringIO()

        try:
            exec(code, self.namespace)
            output = sys.stdout.getvalue()
            return ExecutionResult(success=True, output=output, code=code)
        except Exception as e:
            output = sys.stdout.getvalue()
            return ExecutionResult(success=False, output=output, error=str(e), code=code)
        finally:
            sys.stdout = old_stdout

print("Classe Executor definie : execution securisee de code Python avec capture stdout")

Classe Executor definie : execution securisee de code Python avec capture stdout


## 7. Verifier Agent

In [1]:
class Verifier:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def verify(self, question: str, result: ExecutionResult) -> Tuple[VerificationStatus, str]:
        if not result.success:
            return VerificationStatus.FAILED, f"Erreur d'execution: {result.error}"

        prompt = f"""Verifie si ce resultat repond a la question.

QUESTION: {question}

RESULTAT:
{result.output[:1000]}

Reponds par:
- SUCCESS si le resultat est complet et correct
- NEEDS_REFINEMENT si le resultat est partiel mais prometteur
- FAILED si le resultat ne repond pas du tout

Verdict:"""

        response = self.llm.generate(prompt, temperature=0.1).upper()

        if 'SUCCESS' in response:
            return VerificationStatus.SUCCESS, "Resultat valide"
        elif 'REFINEMENT' in response:
            return VerificationStatus.NEEDS_REFINEMENT, "Necessite des ajustements"
        return VerificationStatus.FAILED, "Resultat insuffisant"

print("Classe Verifier definie : validation des resultats (SUCCESS / NEEDS_REFINEMENT / FAILED)")

Classe Verifier definie : validation des resultats (SUCCESS / NEEDS_REFINEMENT / FAILED)


## 8. DS-STAR Orchestrator

In [1]:
class DSStarAgent:
    def __init__(self, df: pd.DataFrame, max_iterations: int = 3):
        self.llm = LLMClient()
        self.planner = Planner(self.llm)
        self.coder = Coder(self.llm)
        self.executor = Executor(df)
        self.verifier = Verifier(self.llm)
        self.max_iterations = max_iterations

    def analyze(self, question: str, file_context: str = None) -> Dict:
        context = file_context or "DataFrame 'df' disponible"

        for iteration in range(self.max_iterations):
            print(f"\n=== ITERATION {iteration + 1}/{self.max_iterations} ===")

            # Step 1: Plan
            print("[PLANNER] Creation du plan...")
            plan = self.planner.create_plan(question, context)
            print(f"Etapes: {len(plan.steps)}")

            # Step 2: Generate code
            print("[CODER] Generation du code...")
            code = self.coder.generate_code(plan, context)

            # Step 3: Execute
            print("[EXECUTOR] Execution...")
            result = self.executor.execute(code)

            if not result.success:
                print(f"[ERROR] {result.error}")
                context += f"\nErreur precedente: {result.error}\nCorrige le code."
                continue

            print(f"[OUTPUT] {result.output[:200]}...")

            # Step 4: Verify
            print("[VERIFIER] Verification...")
            status, message = self.verifier.verify(question, result)
            print(f"[STATUS] {status.value}: {message}")

            if status == VerificationStatus.SUCCESS:
                return {
                    'success': True,
                    'output': result.output,
                    'code': code,
                    'iterations': iteration + 1
                }

            # Raffinement
            context += f"\nResultat partiel: {result.output[:500]}\nAmeliore l'analyse."

        return {
            'success': False,
            'output': result.output if 'result' in dir() else '',
            'error': 'Max iterations atteint',
            'iterations': self.max_iterations
        }

print("Classe DSStarAgent definie : orchestrateur Planner-Coder-Executor-Verifier avec boucle iterative")

[PLANNER] Creation du plan...


## 9. Test avec un Dataset

In [9]:
# Dataset de test
import tempfile
import os

df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=200, freq='D'),
    'product': np.random.choice(['A', 'B', 'C', 'D'], 200),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 200),
    'revenue': np.random.uniform(100, 5000, 200).round(2),
    'units': np.random.randint(1, 100, 200)
})
print(f'Dataset: {len(df)} lignes')
df.head()

Dataset: 200 lignes


,date,product,region,revenue,units
0,2024-01-01,A,Sud,3203.50,52
1,2024-01-02,D,Nord,2565.75,46
2,2024-01-03,C,Sud,3661.69,97
3,2024-01-04,C,Nord,1883.81,23
4,2024-01-05,D,Sud,732.91,72


Test de l'agent DS-STAR avec une iteration.

In [10]:
# Test de l'agent DS-STAR avec 1 iteration pour rapidite
agent = DSStarAgent(df, max_iterations=1)

question = "Quelle est la region avec le plus grand revenu moyen?"
result = agent.analyze(question)

print("\n" + "="*50)
print("RESULTAT FINAL:")
print("="*50)
if result['success']:
    print(result['output'])
else:
    print(f"Erreur: {result.get('error', 'Inconnu')}")


=== ITERATION 1/1 ===
[PLANNER] Creation du plan...


Etapes: 3
[CODER] Generation du code...

Provider List: https://docs.litellm.ai/docs/providers



[EXECUTOR] Execution...
Provider List: https://docs.litellm.ai/docs/providers


[ERROR] ['revenu']

RESULTAT FINAL:
Erreur: Max iterations atteint


## 10. Resume du Lab
### Ce que nous avons implemente
1. **Planner**: Decompose la question en etapes
2. **Coder**: Genere le code Python
3. **Executor**: Execute en toute securite
4. **Verifier**: Valide ou demande raffinement
### Points cles
- La boucle iterative permet de corriger les erreurs
- Le verifier assure la qualite des resultats
- Le contexte est enrichi a chaque iteration
### Prochaine etape
- **Lab 12**: Workshop DS-STAR complet avec fichiers reels

## Exercice : Analysez vos propres donnees

En utilisant l'architecture DS-STAR que vous venez d'apprendre, creez un agent capable d'analyser un dataset de votre choix.

### Objectifs
1. Creer un dataset personnalise (ou utiliser un dataset publique)
2. Configurer l'agent DS-STAR avec ce dataset
3. Poser 3 questions d'analyse differentes
4. Observer le comportement iteratif de l'agent


In [11]:
# TODO: Creez votre propre dataset
# Exemples : donnees de ventes, meteo, scores sportifs, etc.
mon_dataset = pd.DataFrame({
    # Definissez vos colonnes ici
})

# TODO: Instanciez l'agent DS-STAR
mon_agent = DSStarAgent(mon_dataset, max_iterations=2)

# TODO: Posez 3 questions d'analyse progressives
question_1 = "..."  # Question simple (agregation)
question_2 = "..."  # Question intermediaire (comparaison)
question_3 = "..."  # Question complexe (analyse temporelle ou correlation)

# TODO: Executez et analysez les resultats
resultat_1 = mon_agent.analyze(question_1)
print(f"Reussite: {resultat_1['success']}, Iterations: {resultat_1['iterations']}")



=== ITERATION 1/2 ===
[PLANNER] Creation du plan...


Etapes: 4
Provider List: https://docs.litellm.ai/docs/providers


[CODER] Generation du code...


[EXECUTOR] Execution...
Provider List: https://docs.litellm.ai/docs/providers




[OUTPUT] ==================================================
1. EXPLORATION ET NETTOYAGE DES DONNÉES
Dimensions initiales du DataFrame : 0 lignes, 0 colonnes

...
[VERIFIER] Verification...


[STATUS] failed: Resultat insuffisant
Provider List: https://docs.litellm.ai/docs/providers



=== ITERATION 2/2 ===
[PLANNER] Creation du plan...


Etapes: 4
Provider List: https://docs.litellm.ai/docs/providers


[CODER] Generation du code...


[EXECUTOR] Execution...
Provider List: https://docs.litellm.ai/docs/providers


[OUTPUT] ==================================================
1. DIAGNOSTIC ET CORRECTION DE L'IMPORTATION
Diagnostic : Le DataFrame 'df' initial est vide ou no...
[VERIFIER] Verification...


[STATUS] failed: Resultat insuffisant
Provider List: https://docs.litellm.ai/docs/providers


Reussite: False, Iterations: 2


## Exercice : Amelioration du Prompt du Planner

Le parseur du Planner utilise des expressions regulieres pour extraire les etapes. L'objectif est d'ameliorer le prompt et/ou le parseur pour augmenter le taux de succes de l'extraction.

### Objectifs
1. Analyser pourquoi le Planner extrait parfois 0 etapes
2. Proposer un prompt plus robuste avec des instructions plus strictes
3. Implementer un parseur de fallback (ex: detecter les puces `-` ou `*`)

**Indice :**
- Le format actuel exige `REASONING:` et `STEPS:` avec des numeros `1.`
- Un fallback pourrait chercher des lignes commencant par `- ` ou `* `
- Testez votre version amelioree sur les memes questions que le Planner original

In [12]:
# Exercice : Amelioration du prompt et du parseur du Planner
# Objectif : Rendre l'extraction des etapes plus robuste

# TODO: Analysez le prompt actuel du Planner (cell-8) et identifiez les faiblesses
# Le prompt demande "REASONING:" et "STEPS:" avec des numeros
# Que se passe-t-il si le LLM repond differemment ?

# TODO: Creez une classe RobustPlanner qui herite de Planner
class RobustPlanner(Planner):
    """Planner avec prompt ameliore et parseur de fallback."""
    
    def create_plan(self, question: str, context: str) -> Plan:
        # Etape 1: Modifiez le prompt pour etre plus explicite
        prompt = f"""Tu es un planificateur d'analyse de donnees.
CONTEXTE: {context[:600]}
QUESTION: {question}

IMPORTANT: Reponds OBLIGATOIREMENT dans ce format exact:
REASONING: [ton raisonnement en une phrase]
STEPS:
1. [premiere etape]
2. [deuxieme etape]
3. [troisieme etape]"""
        
        response = self.llm.generate(prompt, temperature=0.3)
        
        # Etape 2: Parse avec le regex standard
        steps = []
        reasoning = ""
        in_steps = False
        for line in response.split('\n'):
            if line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
            elif line.startswith('STEPS:'):
                in_steps = True
            elif in_steps and re.match(r'^\d+\.', line.strip()):
                steps.append(re.sub(r'^\d+\.\s*', '', line.strip()))
        
        # Etape 3: Fallback si aucune etape trouvee
        # Indice: cherchez des lignes commencant par "- " ou "* "
        if not steps:
            pass  # TODO etudiant : implementez le fallback
        
        return Plan(steps=steps, reasoning=reasoning)

# TODO: Testez et comparez avec le Planner original
# robust_planner = RobustPlanner(LLMClient())
# test_question = "Quelle region genere le plus de revenus?"
# test_context = "DataFrame avec colonnes: date, product, region, revenue, units"
# plan = robust_planner.create_plan(test_question, test_context)
# print(f"Etapes trouvees: {len(plan.steps)}")
# for i, step in enumerate(plan.steps):
#     print(f"  {i+1}. {step}")

print("Exercice a completer : amelioration du prompt et parseur du Planner")

Exercice a completer


## Exercice : Analyseur d'Erreurs pour l'Executor

L'Executor retourne des messages d'erreur Python (KeyError, NameError, etc.). L'objectif est de creer un composant `ErrorAnalyzer` qui classifie les erreurs et genere un contexte de correction automatique pour la boucle iterative.

### Objectifs
1. Classifier les erreurs les plus frequentes du code genere par le LLM
2. Creer un dictionnaire de correspondance erreur -> correction
3. Integrer l'ErrorAnalyzer dans la boucle du DSStarAgent

**Indice :**
- Erreurs frequentes : `KeyError` (mauvais nom de colonne), `NameError` (variable non definie), `AttributeError` (methode inexistante)
- Pour chaque type, proposez une correction automatique (ex: suggerer les noms de colonnes proches)
- Utilisez `difflib.get_close_matches()` pour suggerer des noms de colonnes similaires

In [13]:
# Exercice : Analyseur d'erreurs automatique pour l'Executor
# Objectif : Classifier et corriger automatiquement les erreurs du code genere

import difflib

class ErrorAnalyzer:
    """Analyse les erreurs d'execution et suggere des corrections."""
    
    def __init__(self, available_columns: list = None):
        self.available_columns = available_columns or []
    
    def classify_error(self, error_message: str) -> str:
        """
        Classifie le type d'erreur.
        
        Args:
            error_message: message d'erreur Python
            
        Returns:
            Type d'erreur: 'key_error', 'name_error', 'attribute_error', 'type_error', 'other'
        """
        # TODO: Implementez la classification
        # Indice: utilisez des mots-cles comme "KeyError", "NameError", etc.
        error_lower = error_message.lower()
        
        if 'keyerror' in error_lower:
            return 'key_error'
        # TODO etudiant : ajoutez les autres types d'erreurs
        
        return 'other'
    
    def suggest_correction(self, error_message: str, error_type: str) -> str:
        """
        Suggere une correction basee sur le type d'erreur.
        
        Args:
            error_message: message d'erreur original
            error_type: type classifie
            
        Returns:
            Message de correction pour enrichir le contexte
        """
        if error_type == 'key_error':
            # Etape 1: Extrayez le nom de colonne errone
            # Indice: cherchez entre guillemets dans le message d'erreur
            wrong_col = None  # TODO etudiant : extrayez avec un regex
            
            # Etape 2: Suggerez des colonnes proches
            # Indice: difflib.get_close_matches(wrong_col, self.available_columns, n=3)
            suggestions = []
            
            return f"La colonne '{wrong_col}' n'existe pas. Colonnes disponibles: {self.available_columns}. Suggestions proches: {suggestions}"
        
        # TODO etudiant : ajoutez des corrections pour name_error, attribute_error, etc.
        return f"Erreur de type {error_type}: {error_message}"

# TODO: Testez l'ErrorAnalyzer
# analyzer = ErrorAnalyzer(available_columns=['date', 'product', 'region', 'revenue', 'units'])
# 
# # Test avec une erreur KeyError typique
# error = "KeyError: 'revenu'"
# error_type = analyzer.classify_error(error)
# correction = analyzer.suggest_correction(error, error_type)
# print(f"Type: {error_type}")
# print(f"Correction: {correction}")

print("Exercice a completer : analyseur d'erreurs automatique pour l'Executor")

Exercice a completer


### Points de reflexion
- Combien d'iterations sont necessaires pour chaque type de question ?
- Quelles erreurs l'agent rencontre-t-il et comment les corrige-t-il ?
- Comment pourriez-vous ameliorer le verifier ?


## Références

1. S. Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, arXiv:2210.03629, ICLR 2023. Paradigme Reasoning+Acting : l'agent alterne raisonnement et actions observables — fondement de la boucle Planner-Coder-Verifier.
2. N. Shinn et al., *Reflexion: Language Agents with Verbal Reinforcement Learning*, arXiv:2303.11366, NeurIPS 2023. Auto-réflexion verbale et raffinement itératif après échec — principe de l'étape Verifier → Coder.
3. Guo et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Data*, arXiv:2509.21825, 2025. Agent DS-STAR dont ce lab implémente la boucle cœur (suite Lab 10).
4. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2025. Cadre conceptuel des agents LLM (suite Labs 8-10).